# Questão 5 — Dimensão de Calendário

## Objetivo

Calcular a média de vendas das lojas físicas por dia da semana considerando
também os dias em que a loja esteve aberta, mas não registrou nenhuma venda.

Uma agregação realizada diretamente sobre `orders` excluiria automaticamente
os dias sem pedidos, podendo superestimar a média de determinados dias da
semana.

Para evitar esse viés, será construída uma dimensão de datas contendo todos
os dias do período analisado. Essa dimensão será relacionada às vendas diárias
por meio de um `LEFT JOIN`, atribuindo valor zero aos dias sem vendas.

In [ ]:
from pathlib import Path

import duckdb


PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

conn = duckdb.connect()

orders_path = RAW_DATA_DIR / "orders.csv"

conn.execute(
    f"""
    CREATE OR REPLACE TABLE orders AS
    SELECT *
    FROM read_csv_auto('{orders_path.as_posix()}');
    """
)

In [2]:
query_period = """
SELECT
    CAST(MIN(placed_at) AS DATE) AS data_inicial,
    CAST(MAX(placed_at) AS DATE) AS data_final
FROM orders;
"""

analysis_period = conn.execute(query_period).df()

analysis_period

,data_inicial,data_final
0,2020-01-01,2026-12-31


In [3]:
query_calendar = """
CREATE OR REPLACE TEMP TABLE dim_calendar AS

WITH date_range AS (
    SELECT
        CAST(MIN(placed_at) AS DATE) AS min_date,
        CAST(MAX(placed_at) AS DATE) AS max_date
    FROM orders
)

SELECT
    CAST(calendar_date AS DATE) AS date,
    EXTRACT(YEAR FROM calendar_date) AS year,
    EXTRACT(MONTH FROM calendar_date) AS month,
    EXTRACT(DAY FROM calendar_date) AS day,
    EXTRACT(ISODOW FROM calendar_date) AS weekday_number,
    CASE EXTRACT(ISODOW FROM calendar_date)
        WHEN 1 THEN 'Segunda-feira'
        WHEN 2 THEN 'Terça-feira'
        WHEN 3 THEN 'Quarta-feira'
        WHEN 4 THEN 'Quinta-feira'
        WHEN 5 THEN 'Sexta-feira'
        WHEN 6 THEN 'Sábado'
        WHEN 7 THEN 'Domingo'
    END AS weekday_name
FROM date_range,
     GENERATE_SERIES(
         min_date,
         max_date,
         INTERVAL 1 DAY
     ) AS dates(calendar_date);
"""

conn.execute(query_calendar)

In [4]:
query_calendar_validation = """
SELECT
    COUNT(*) AS total_dias,
    MIN(date) AS data_inicial,
    MAX(date) AS data_final,
    COUNT(DISTINCT date) AS datas_distintas
FROM dim_calendar;
"""

calendar_validation = conn.execute(query_calendar_validation).df()

calendar_validation

,total_dias,data_inicial,data_final,datas_distintas
0,2557,2020-01-01,2026-12-31,2557


In [5]:
query_daily_sales = """
SELECT
    CAST(placed_at AS DATE) AS date,
    SUM(total) AS vendas_diarias
FROM orders
WHERE channel = 'pos'
GROUP BY CAST(placed_at AS DATE)
ORDER BY date;
"""

daily_sales = conn.execute(query_daily_sales).df()

daily_sales.head()

,date,vendas_diarias
0,2020-01-01,420574.61
1,2020-01-02,284813.10
2,2020-01-03,192399.41
3,2020-01-04,160525.49
4,2020-01-05,222574.28


In [6]:
query_calendar_sales = """
SELECT
    c.date,
    c.weekday_number,
    c.weekday_name,
    COALESCE(SUM(o.total), 0) AS vendas_diarias
FROM dim_calendar AS c
LEFT JOIN orders AS o
    ON CAST(o.placed_at AS DATE) = c.date
    AND o.channel = 'pos'
GROUP BY
    c.date,
    c.weekday_number,
    c.weekday_name
ORDER BY c.date;
"""

calendar_sales = conn.execute(query_calendar_sales).df()

calendar_sales.head()

,date,weekday_number,weekday_name,vendas_diarias
0,2020-01-01,3,Quarta-feira,420574.61
1,2020-01-02,4,Quinta-feira,284813.10
2,2020-01-03,5,Sexta-feira,192399.41
3,2020-01-04,6,Sábado,160525.49
4,2020-01-05,7,Domingo,222574.28


In [ ]:
query_weekday_average = """
WITH daily_sales AS (
    SELECT
        c.date,
        c.weekday_number,
        c.weekday_name,
        COALESCE(SUM(o.total), 0) AS vendas_diarias
    FROM dim_calendar AS c
    LEFT JOIN orders AS o
        ON CAST(o.placed_at AS DATE) = c.date
        AND o.channel = 'pos'
    GROUP BY
        c.date,
        c.weekday_number,
        c.weekday_name
)

SELECT
    weekday_name AS dia_semana,
    COUNT(*) AS dias_no_calendario,
    SUM(CASE WHEN vendas_diarias = 0 THEN 1 ELSE 0 END) AS dias_sem_venda,
    ROUND(AVG(vendas_diarias), 2) AS media_vendas
FROM daily_sales
GROUP BY
    weekday_number,
    weekday_name
ORDER BY media_vendas ASC;
"""

weekday_average = conn.execute(query_weekday_average).df()

weekday_average

weekday_average.to_csv(
    OUTPUT_DIR / "weekday_sales.csv",
    index=False,
)

print("weekday_sales.csv exportado com sucesso.")

,dia_semana,dias_no_calendario,dias_sem_venda,media_vendas
0,Quinta-feira,366,20.0,157154.32
1,Domingo,365,12.0,157616.13
2,Segunda-feira,365,7.0,158241.15
3,Sábado,365,11.0,164858.27
4,Terça-feira,365,8.0,166118.83
5,Sexta-feira,365,10.0,170193.68
6,Quarta-feira,366,10.0,173605.44
